In [25]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np

In [2]:
def get_dests(raw_data_paths, competition, seasons):
    """
    Get the destination paths for each season.
    """
    return [f"{raw_data_paths}/{competition}/{season}" for season in seasons]

In [3]:
def scrape_competitions_for_seasons(all_seasons, raw_data_path, competitions, urls):
    for competition in competitions:
        for season, dest in zip(all_seasons, get_dests(raw_data_path, competition, all_seasons)):
            url = urls[competition][season]
            print(f"Scraping {url} to {dest}")
            # Run the scraper
            #scrape_season_match_data(url, dest)

In [4]:
def get_processed_path(processed_data_path, competition):
    """
    Get the processed data path for a given competition.
    """
    return f"{processed_data_path}/{competition}"

In [5]:
def check_if_target_df_exists(processed_path, all_matches_df):
    """
    Check if the target DataFrame already exists and matches the expected length.
    """
    target_path = f"{processed_path}/all_target_df.csv"
    if os.path.exists(target_path):
        df = pd.read_csv(target_path)
        return len(df) == len(all_matches_df)
    return False

def check_if_data_df_exists(processed_path, all_matches_df):
    """
    Check if the data DataFrame already exists and matches the expected length.
    """
    data_path = f"{processed_path}/all_data_df.csv"
    if os.path.exists(data_path):
        df = pd.read_csv(data_path)
        return len(df) == len(all_matches_df)
    return False

In [6]:
def process_raw_match_data(all_competitions, all_seasons, raw_data_path, processed_data_path, target_columns, verbose=True):
    for competition in all_competitions:
        for season in all_seasons:
            season_path = f'{raw_data_path}/{competition}/{season}/'
            processed_path = f'{get_processed_path(processed_data_path, competition)}/{season}/'
            if not os.path.exists(processed_path):
                os.makedirs(processed_path)
            all_matches = pd.read_csv(f'{season_path}/all_matches.csv').iloc[:, 1].tolist()
            # Check if target df exists and matches length
            if check_if_target_df_exists(processed_path, all_matches):
                if verbose:
                    print(f"Target DataFrame for {competition} {season} already exists and matches. Skipping.")
            else:
                if verbose:
                    print(f"Processing target DataFrame for {competition} {season}.")
                target_df = []
                for match_name in all_matches:
                    home_df = pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
                    away_df = pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
                    match_df = pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
                    target_df.append(process_match_target_var(home_df, away_df, match_df, match_name))
                target_df = pd.concat(target_df, axis=0)
                target_df.to_csv(f'{processed_path}/all_target_df.csv', index=False)

        for season in all_seasons:
            season_path = f'{raw_data_path}/{competition}/{season}/'
            processed_path = f'{get_processed_path(processed_data_path, competition)}/{season}/'
            all_matches = pd.read_csv(f'{season_path}/all_matches.csv').iloc[:, 1].tolist()
            # Check if data df exists and matches length
            if check_if_data_df_exists(processed_path, all_matches):
                if verbose:
                    print(f"Data DataFrame for {competition} {season} already exists and matches. Skipping.")
            else:
                if verbose:
                    print(f"Processing data DataFrame for {competition} {season}.")
                data_df = []
                for match_name in all_matches:
                    home_df = pd.read_csv(f'{season_path}/{match_name}_home_player_stat.csv')
                    away_df = pd.read_csv(f'{season_path}/{match_name}_away_player_stat.csv')
                    match_df = pd.read_csv(f'{season_path}/{match_name}_match_stat.csv', index_col=0)
                    data_df.append(process_match_other_var(home_df, away_df, match_df, match_name, target_columns))
                data_df = pd.concat(data_df, axis=0)
                data_df.to_csv(f'{processed_path}/all_data_df.csv', index=False)

In [7]:
f'{RAW_DATA_PATH}/{COMPETITIONS[0]}/{ALL_SEASONS[0]}/'

'/Users/tianqihuang/Documents/GitHub/betbot//data/raw//premier_league/2017-18/'

In [8]:
COMPETITIONS

('premier_league',)

In [9]:
process_raw_match_data(COMPETITIONS, ALL_SEASONS, RAW_DATA_PATH, PROCESSED_DATA_PATH, TARGET_COLUMNS)

Target DataFrame for premier_league 2017-18 already exists and matches. Skipping.
Target DataFrame for premier_league 2018-19 already exists and matches. Skipping.
Target DataFrame for premier_league 2019-20 already exists and matches. Skipping.
Target DataFrame for premier_league 2020-21 already exists and matches. Skipping.
Target DataFrame for premier_league 2021-22 already exists and matches. Skipping.
Target DataFrame for premier_league 2022-23 already exists and matches. Skipping.
Target DataFrame for premier_league 2023-24 already exists and matches. Skipping.
Target DataFrame for premier_league 2024-25 already exists and matches. Skipping.
Data DataFrame for premier_league 2017-18 already exists and matches. Skipping.
Data DataFrame for premier_league 2018-19 already exists and matches. Skipping.
Data DataFrame for premier_league 2019-20 already exists and matches. Skipping.
Data DataFrame for premier_league 2020-21 already exists and matches. Skipping.
Data DataFrame for premi

In [10]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"

In [11]:
encoder = TeamEncoder.load(team_encoder_path)

In [12]:
competition = COMPETITIONS[0]

In [13]:
seasons=sorted(ALL_SEASONS)
data_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons]
target_dfs=[pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]

In [14]:
test_df=pd.DataFrame({
    'home': ['Chelsea'],
    'away': ['Bournemouth'],
    'date': ['2024-05-19']
    })

In [15]:
CURRENT_SEASON

'2024-25'

In [16]:
# Prepare season_dfs as a dict for transform_spot
season_dfs_dict = {season: df for season, df in zip(seasons, data_dfs)}
# Run transform_spot for the current season and test_df
spot_features = encoder.transform_spot(season_dfs_dict, test_df)
spot_features

,home,away,date,encoded_home_Arsenal,encoded_home_Bournemouth,encoded_home_Brighton,encoded_home_Burnley,encoded_home_Chelsea,encoded_home_Crystal Palace,encoded_home_Everton,...,encoded_away_Manchester Utd,encoded_away_Newcastle Utd,encoded_away_Southampton,encoded_away_Stoke City,encoded_away_Swansea City,encoded_away_Tottenham,encoded_away_Watford,encoded_away_West Brom,encoded_away_West Ham,encoded_away_new_team_away
0,Chelsea,Bournemouth,2024-05-19,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
already_feature_df= pd.read_csv(f"../../data/features/premier_league/all_combined_features_2017-24.csv")

In [18]:
already_feature_df['date'].max()

'2025-05-25'

In [19]:
already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), [col for col in already_feature_df.columns if 'encoded' in col]].iloc[-1].to_numpy()==spot_features.drop(columns=['home', 'away', 'date']).iloc[0].to_numpy()

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True])

In [20]:
previous_season_feature= PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
spot_features=previous_season_feature.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

In [21]:
spot_features

,home,away,date,prev_season_avg_home_home_performance_pk,prev_season_avg_home_home_performance_pkatt,prev_season_avg_home_home_performance_crdr,prev_season_avg_home_home_performance_touches,prev_season_avg_home_home_performance_tkl,prev_season_avg_home_home_performance_int,prev_season_avg_home_home_performance_blocks,...,prev_season_avg_away_away_performance_tklw,prev_season_avg_away_away_performance_pkwon,prev_season_avg_away_away_performance_pkcon,prev_season_avg_away_away_performance_og,prev_season_avg_away_away_performance_recov,prev_season_avg_away_away_aerial_duels_won,prev_season_avg_away_away_aerial_duels_lost,prev_season_avg_away_away_aerial_duels_won%,prev_season_avg_away_days_ago,prev_season_avg_away_weights
0,Chelsea,Bournemouth,2024-05-19,0.491007,0.491007,1.315229e-33,660.53246,19.901075,5.499999,12.410068,...,12.999997,0.0,3.187934e-59,5.038279e-120,61.49999,15.499999,14.0,55.599996,4.000008,0.500168


In [24]:
already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), [col for col in already_feature_df.columns if 'prev_season' in col]].iloc[-2]

prev_season_avg_home_home_performance_pk         4.910069e-01
prev_season_avg_home_home_performance_pkatt      4.910069e-01
prev_season_avg_home_home_performance_crdr       1.315229e-33
prev_season_avg_home_home_performance_touches    6.605325e+02
prev_season_avg_home_home_performance_tkl        1.990107e+01
                                                     ...     
prev_season_avg_away_away_aerial_duels_won       1.550000e+01
prev_season_avg_away_away_aerial_duels_lost      1.400000e+01
prev_season_avg_away_away_aerial_duels_won%      5.560000e+01
prev_season_avg_away_days_ago                    4.000008e+00
prev_season_avg_away_weights                     5.001676e-01
Name: 2161, Length: 432, dtype: float64

In [26]:
np.isclose(already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), [col for col in already_feature_df.columns if 'prev_season' in col]].iloc[-2].to_numpy(), spot_features.drop(columns=['home', 'away', 'date']).iloc[0].to_numpy())

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [28]:
generator = TeamLagFeatureGenerator(lookback=5)
spot_features = generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

In [29]:
spot_features

,home,away,date,home_lag1_home_performance_pk,home_lag1_home_performance_pkatt,home_lag1_home_performance_crdr,home_lag1_home_performance_touches,home_lag1_home_performance_tkl,home_lag1_home_performance_int,home_lag1_home_performance_blocks,...,away_lag5_away_performance_int.1,away_lag5_away_performance_tklw,away_lag5_away_performance_pkwon,away_lag5_away_performance_pkcon,away_lag5_away_performance_og,away_lag5_away_performance_recov,away_lag5_away_aerial_duels_won,away_lag5_away_aerial_duels_lost,away_lag5_away_aerial_duels_won%,away_lag5_was_home
0,Chelsea,Bournemouth,2024-05-19,0,0,0,701,17,5,6,...,5,10,0,0,0,39,26,24,52.0,1


In [32]:
already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), [col for col in already_feature_df.columns if 'home_lag' in col or 'away_lag' in col]].iloc[-2]

home_lag1_home_performance_pk           0.0
home_lag1_home_performance_pkatt        0.0
home_lag1_home_performance_crdr         0.0
home_lag1_home_performance_touches    701.0
home_lag1_home_performance_tkl         17.0
                                      ...  
away_lag_5_away_cards                   2.0
away_lag_5_home_shots                  24.0
away_lag_5_away_shots                   8.0
away_lag_5_home_sots                    9.0
away_lag_5_away_sots                    4.0
Name: 2161, Length: 2250, dtype: float64

In [35]:
np.isclose(already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), spot_features.drop(columns=['home', 'away', 'date']).columns].iloc[-2].to_numpy(), spot_features.drop(columns=['home', 'away', 'date']).iloc[0].to_numpy()).all()

True

In [36]:
team_rest_days_calculator = TeamRestDaysCalculator()
spot_features = team_rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

In [37]:
np.isclose(already_feature_df.loc[(already_feature_df['home'] == 'Chelsea') & (already_feature_df['away'] == 'Bournemouth'), spot_features.drop(columns=['home', 'away', 'date']).columns].iloc[-2].to_numpy(), spot_features.drop(columns=['home', 'away', 'date']).iloc[0].to_numpy()).all()

True